# Introduction to PyTorch (Beginner-friendly)

**Learning Objectives:**
- Understand what PyTorch is and why it's the go-to framework for deep learning
- Create and manipulate tensors — PyTorch's core data structure
- Master tensor operations, broadcasting, and reshaping
- Convert seamlessly between NumPy and PyTorch
- Leverage GPU acceleration for faster computation
- Understand automatic differentiation (autograd) — the engine behind all learning

**Prerequisites:** Python basics, NumPy fundamentals (complete NumPy notebook first)

**Estimated Time:** ~60 minutes

---

PyTorch is the most widely used deep learning framework in research and increasingly in production. If NumPy is your calculator, **PyTorch is your calculator that can also learn from its mistakes.** It extends NumPy-like tensor operations with two superpowers: **GPU acceleration** and **automatic differentiation**.

**Why PyTorch?** Every modern deep learning breakthrough — GPT, Stable Diffusion, AlphaFold — was built with PyTorch or similar frameworks. Understanding PyTorch is essential for:
- Representing and transforming data for deep learning pipelines
- Running computations on GPUs for massive speedups
- Computing gradients automatically (no more manual calculus!)
- Building and experimenting with custom ML ideas

**Learning Path:** This notebook builds on your NumPy knowledge (tensors ≈ arrays). We focus on **PyTorch fundamentals** here — tensors, operations, GPU usage, and autograd. Neural networks, training loops, and model building will be covered in the next session.

**🎯 Success Indicators:** By the end, you should be able to:
- Create tensors and check their shape, dtype, and device
- Perform arithmetic, matrix multiplication, and broadcasting
- Convert between NumPy arrays and PyTorch tensors
- Move tensors to GPU and understand when it helps
- Use autograd to compute gradients automatically
- Implement gradient descent from scratch using autograd

**💡 Beginner Tips:**
- If you know NumPy, you already know 60% of PyTorch — tensors work almost identically
- Don't worry about GPUs at first — everything runs on CPU too
- Read error messages carefully — most PyTorch errors are about tensor shapes or device mismatches
- Run each cell and experiment — change values, break things, learn!

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ============================================================
# 🔧 ENVIRONMENT CHECK
# ============================================================
print("=" * 60)
print("🔧 PYTORCH ENVIRONMENT SETUP")
print("=" * 60)

print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ NumPy version:   {np.__version__}")

# Device setup — PyTorch can run on CPU or GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device:          {device}")

if torch.cuda.is_available():
    print(f"   GPU Name:        {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("   💡 No GPU detected — running on CPU (totally fine for learning!)")

print(f"\n{'=' * 60}")
print("📊 PYTORCH vs NUMPY — QUICK COMPARISON")
print("=" * 60)
print(f"{'Feature':<30} {'NumPy':>12} {'PyTorch':>12}")
print("-" * 54)
print(f"{'Core data structure':<30} {'ndarray':>12} {'Tensor':>12}")
print(f"{'GPU support':<30} {'❌':>12} {'✅':>12}")
print(f"{'Auto differentiation':<30} {'❌':>12} {'✅':>12}")
print(f"{'Neural network layers':<30} {'❌':>12} {'✅':>12}")
print(f"{'Optimizers built-in':<30} {'❌':>12} {'✅':>12}")
print(f"{'Interop with each other':<30} {'✅':>12} {'✅':>12}")
print("\n💡 Think of PyTorch as 'NumPy with superpowers for learning'")

---

# Part 1: Tensors — The Building Blocks 🧱

## What is a Tensor?

A **tensor** is just a multi-dimensional array — exactly like a NumPy `ndarray`. The name sounds fancy, but you already know what they are:

| Dimensions | Math Name | NumPy | PyTorch | Example |
|:---:|:---:|:---:|:---:|:---|
| 0 | Scalar | `np.float64(3.14)` | `torch.tensor(3.14)` | A single temperature reading |
| 1 | Vector | `np.array([1,2,3])` | `torch.tensor([1,2,3])` | A row of pixel values |
| 2 | Matrix | `np.array([[1,2],[3,4]])` | `torch.tensor([[1,2],[3,4]])` | A grayscale image |
| 3 | 3D Tensor | `np.zeros((3,28,28))` | `torch.zeros(3,28,28)` | A color image (RGB) |
| 4 | 4D Tensor | `np.zeros((32,3,28,28))` | `torch.zeros(32,3,28,28)` | A batch of color images |

**The key difference:** PyTorch tensors can track their own gradients and run on GPUs. That's what makes deep learning possible.

In [ ]:
# ============================================================
# 🧱 CREATING TENSORS — MANY WAYS TO BUILD THEM
# ============================================================
print("=" * 60)
print("🧱 CREATING TENSORS")
print("=" * 60)

# --- From Python lists (most intuitive) ---
print("\n📌 Method 1: From Python lists")
t1 = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"   tensor:  {t1}")
print(f"   shape:   {t1.shape}")
print(f"   dtype:   {t1.dtype}")

t2 = torch.tensor([[1, 2, 3],
                    [4, 5, 6]])
print(f"\n   2D tensor:\n   {t2}")
print(f"   shape: {t2.shape}  →  (2 rows, 3 columns)")

# --- Special constructors (like NumPy!) ---
print(f"\n{'─' * 60}")
print("📌 Method 2: Special constructors")
zeros = torch.zeros(3, 4)       # All zeros — useful for initialization
ones  = torch.ones(2, 3)        # All ones
rand  = torch.rand(3, 3)        # Uniform [0, 1) — good for random weights
randn = torch.randn(3, 3)      # Normal(0, 1) — standard for weight init
arange = torch.arange(0, 10, 2) # Like Python range()
eye   = torch.eye(3)            # Identity matrix
full  = torch.full((2, 3), 7.0) # Filled with a specific value

print(f"   zeros(3,4) shape: {zeros.shape}")
print(f"   randn(3,3):\n   {randn}")
print(f"   arange(0,10,2): {arange}")
print(f"   eye(3):\n   {eye}")
print(f"   full(2,3, fill=7):\n   {full}")

# --- Tensor attributes (the 3 things you always check) ---
print(f"\n{'─' * 60}")
print("📌 The 3 attributes you'll check constantly:")
x = torch.randn(32, 3, 28, 28)  # Simulating a batch of 32 RGB images
print(f"   x.shape  = {x.shape}   ← (batch, channels, height, width)")
print(f"   x.dtype  = {x.dtype}       ← float32 is the default for neural nets")
print(f"   x.device = {x.device}         ← CPU or GPU")
print(f"\n   💡 90% of PyTorch debugging is checking these three things!")

# --- Specifying dtypes ---
print(f"\n{'─' * 60}")
print("📌 Common dtypes")
int_tensor   = torch.tensor([1, 2, 3])                    # int64 by default
float_tensor = torch.tensor([1.0, 2.0, 3.0])              # float32 by default
explicit     = torch.tensor([1, 2, 3], dtype=torch.float32) # force float32
converted    = int_tensor.float()                           # cast to float32
print(f"   integers:  dtype = {int_tensor.dtype}")
print(f"   floats:    dtype = {float_tensor.dtype}")
print(f"   explicit:  dtype = {explicit.dtype}")
print(f"   converted: dtype = {converted.dtype}")
print(f"\n   💡 Neural nets need float32 — use .float() to convert if needed")

### ✍️ TODO 1: Tensor Creation Practice

Complete the exercises below to test your understanding of tensor creation.

In [ ]:
# ============================================================
# TODO 1: TENSOR CREATION EXERCISES
# ============================================================

# Exercise 1a: Create a 1D tensor with values [10, 20, 30, 40, 50]
# Your code here:
# t1a = ...


# Exercise 1b: Create a 3x3 tensor filled with the value 5.0
# Your code here:
# t1b = ...


# Exercise 1c: Create a 4x4 identity matrix
# Your code here:
# t1c = ...


# Exercise 1d: Create a tensor of shape (2, 5) with random values from a
#              normal distribution, and print its shape, dtype, and device
# Your code here:
# t1d = ...


# Exercise 1e: Create an integer tensor [1, 2, 3] and convert it to float32
# Your code here:
# t1e_int = ...
# t1e_float = ...


# -------------------------------------------------------------------
# ✅ Uncomment the lines below to check your answers:
# -------------------------------------------------------------------
# print(f"1a: {t1a}")
# print(f"1b:\n{t1b}")
# print(f"1c:\n{t1c}")
# print(f"1d: shape={t1d.shape}, dtype={t1d.dtype}, device={t1d.device}")
# print(f"1e: int dtype={t1e_int.dtype}, float dtype={t1e_float.dtype}")

---

# Part 2: Tensor Operations — Computing with Tensors 🔧

Just like NumPy, PyTorch supports element-wise operations, matrix multiplication, and reductions. If you've used NumPy, this will feel very familiar — most function names are identical.

**Key insight:** Every computation you do with PyTorch tensors can (optionally) be tracked for automatic gradient computation. That's the magic that connects "doing math" to "learning from data."

In [ ]:
# ============================================================
# 🔧 ELEMENT-WISE OPERATIONS & REDUCTIONS
# ============================================================
print("=" * 60)
print("🔧 TENSOR OPERATIONS")
print("=" * 60)

a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([10.0, 20.0, 30.0, 40.0])

# --- Element-wise operations ---
print("\n📌 Element-wise operations (same as NumPy)")
print(f"   a + b  = {a + b}")
print(f"   a * b  = {a * b}")
print(f"   a ** 2 = {a ** 2}")
print(f"   torch.sqrt(a) = {torch.sqrt(a)}")
print(f"   torch.exp(a)  = {torch.exp(a)}")
print(f"   torch.log(a)  = {torch.log(a)}")

# --- Comparison operations ---
print(f"\n📌 Comparison (returns boolean tensors)")
print(f"   a > 2   = {a > 2}")
print(f"   a == 3  = {a == 3}")
print(f"   (a > 2).sum() = {(a > 2).sum()}  ← count of True values")

# --- Reduction operations ---
print(f"\n{'─' * 60}")
print("📌 Reduction operations")
x = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
print(f"   x =\n   {x}")
print(f"   x.sum()         = {x.sum()}")
print(f"   x.mean()        = {x.mean()}")
print(f"   x.std()         = {x.std():.4f}")
print(f"   x.min()         = {x.min()}")
print(f"   x.max()         = {x.max()}")

# --- dim argument — controls WHICH dimension gets collapsed ---
print(f"\n📌 The 'dim' argument (this trips up beginners!)")
print(f"   x.sum(dim=0) = {x.sum(dim=0)}   ← sum DOWN rows → result per column")
print(f"   x.sum(dim=1) = {x.sum(dim=1)}      ← sum ACROSS columns → result per row")
print(f"   x.mean(dim=0) = {x.mean(dim=0)} ← mean DOWN rows")
print(f"   x.mean(dim=1) = {x.mean(dim=1)}    ← mean ACROSS columns")
print(f"\n   💡 dim=0 collapses rows, dim=1 collapses columns")
print(f"   Think: dim=X means 'combine along axis X'")

In [ ]:
# ============================================================
# ✖️ MATRIX MULTIPLICATION
# ============================================================
print("=" * 60)
print("✖️ MATRIX MULTIPLICATION — THE HEART OF DEEP LEARNING")
print("=" * 60)

# --- Basic matmul ---
print("\n📌 Matrix multiplication: A @ B  or  torch.matmul(A, B)")
A = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0],
                   [5.0, 6.0]])    # 3×2
B = torch.tensor([[7.0, 8.0, 9.0],
                   [10.0, 11.0, 12.0]])  # 2×3
C = A @ B                              # 3×3 result
print(f"   A shape: {A.shape}")
print(f"   B shape: {B.shape}")
print(f"   A @ B shape: {C.shape}  ← (3×2) @ (2×3) = (3×3)")
print(f"   Result:\n   {C}")

# --- Why it matters ---
print(f"\n{'─' * 60}")
print("📌 Why matrix multiplication matters for ML:")
print("   Every neural network layer is basically: output = input @ weights + bias")
print()
batch = torch.randn(32, 784)         # 32 flattened images, each 784 pixels
weights = torch.randn(784, 128)      # connection weights: 784 inputs → 128 neurons
bias = torch.randn(128)              # one bias per output neuron
output = batch @ weights + bias      # this IS a neural network layer!
print(f"   input:   {batch.shape}   ← 32 images, 784 features each")
print(f"   weights: {weights.shape}  ← connecting 784 inputs to 128 neurons")
print(f"   bias:    {bias.shape}      ← one per output neuron")
print(f"   output:  {output.shape}   ← 32 images, 128 features each")
print(f"\n   💡 This is exactly what nn.Linear(784, 128) does internally!")

# --- Dot product (1D) ---
print(f"\n{'─' * 60}")
print("📌 Dot product (1D vectors)")
v1 = torch.tensor([1.0, 2.0, 3.0])
v2 = torch.tensor([4.0, 5.0, 6.0])
dot = torch.dot(v1, v2)
print(f"   v1 = {v1}")
print(f"   v2 = {v2}")
print(f"   dot product = 1×4 + 2×5 + 3×6 = {dot.item()}")

### ✍️ TODO 2: Tensor Operations Practice

In [ ]:
# ============================================================
# TODO 2: TENSOR OPERATIONS EXERCISES
# ============================================================

# Exercise 2a: Create two tensors of shape (3, 3) with random values.
#              Compute their element-wise product AND matrix product.
#              Print both results and their shapes.
# Your code here:


# Exercise 2b: Given the tensor below, compute:
#              - The mean of each row (dim=1)
#              - The max value in each column (dim=0)
#              - The index of the max value in each row (argmax, dim=1)
data = torch.tensor([[4.0, 1.0, 7.0],
                      [2.0, 8.0, 3.0],
                      [5.0, 6.0, 0.0]])
# Your code here:


# Exercise 2c: Create a (5, 3) matrix and a (3, 2) matrix with random values.
#              Multiply them and verify the output shape is (5, 2).
# Your code here:


# Exercise 2d: Given a tensor of exam scores, find how many scores are above 70.
scores = torch.tensor([85.0, 42.0, 73.0, 91.0, 58.0, 67.0, 79.0, 95.0])
# Your code here:
# above_70 = ...
# count = ...


# -------------------------------------------------------------------
# ✅ Uncomment to check:
# -------------------------------------------------------------------
# print(f"2d: {count} scores above 70")

---

# Part 3: Broadcasting & Reshaping 📐

## Broadcasting

Broadcasting is how PyTorch (and NumPy) automatically handles operations between tensors of different shapes. Instead of manually copying data to match shapes, PyTorch "stretches" the smaller tensor to fit.

**Broadcasting Rules (same as NumPy):**
1. Compare shapes from right to left
2. Dimensions match if they're equal OR one of them is 1
3. A dimension of size 1 is "stretched" to match the other

## Reshaping

Reshaping changes how a tensor's data is organized without changing the data itself. This is critical in ML — you constantly need to reshape data to match what layers expect.

In [ ]:
# ============================================================
# 📐 BROADCASTING — AUTOMATIC SHAPE MATCHING
# ============================================================
print("=" * 60)
print("📐 BROADCASTING")
print("=" * 60)

# --- Example 1: Adding a row vector to every row of a matrix ---
print("\n📌 Example 1: Matrix + Row Vector")
matrix = torch.tensor([[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0],
                        [7.0, 8.0, 9.0]])        # shape (3, 3)
row = torch.tensor([10.0, 20.0, 30.0])           # shape (3,)
result = matrix + row                              # (3,3) + (3,) → row added to each row
print(f"   matrix (3×3):\n   {matrix}")
print(f"   row    (3,):  {row}")
print(f"   result (3×3):\n   {result}")
print(f"   → The row vector was 'copied' to match each row!")

# --- Example 2: Adding a column vector to every column ---
print(f"\n{'─' * 60}")
print("📌 Example 2: Matrix + Column Vector")
col = torch.tensor([[100.0],
                     [200.0],
                     [300.0]])                    # shape (3, 1)
result2 = matrix + col                            # (3,3) + (3,1) → col added to each column
print(f"   col (3×1):\n   {col}")
print(f"   result (3×3):\n   {result2}")
print(f"   → The column vector was 'stretched' across all columns!")

# --- Example 3: Normalizing data (very common in ML!) ---
print(f"\n{'─' * 60}")
print("📌 Example 3: Real ML use — normalizing features")
# Dataset: 4 samples, 3 features each
data = torch.tensor([[170.0, 65.0, 25.0],
                      [180.0, 80.0, 30.0],
                      [160.0, 55.0, 22.0],
                      [175.0, 70.0, 28.0]])        # (4, 3)
mean = data.mean(dim=0)                            # mean per feature → (3,)
std  = data.std(dim=0)                             # std per feature  → (3,)
normalized = (data - mean) / std                   # broadcasts (4,3) - (3,) / (3,)

print(f"   Raw data (4 samples × 3 features):\n   {data}")
print(f"   Feature means: {mean}")
print(f"   Feature stds:  {std}")
print(f"   Normalized:\n   {normalized}")
print(f"   Normalized means: {normalized.mean(dim=0)}")
print(f"\n   💡 This is standard feature normalization — broadcasting does the heavy lifting!")

In [ ]:
# ============================================================
# 🔄 RESHAPING TENSORS
# ============================================================
print("=" * 60)
print("🔄 RESHAPING TENSORS")
print("=" * 60)

x = torch.arange(12).float()
print(f"\n📌 Original: shape {x.shape} → {x}")

# --- view / reshape ---
print(f"\n📌 view() and reshape() — rearrange into new shape")
r1 = x.view(3, 4)
print(f"   x.view(3, 4):\n   {r1}")
r2 = x.view(2, 2, 3)
print(f"   x.view(2, 2, 3):\n   {r2}")

# Using -1 to auto-compute one dimension
r3 = x.view(4, -1)  # -1 means "figure it out" → 12/4 = 3
print(f"\n   x.view(4, -1) → shape {r3.shape}  (-1 becomes 3)")
r4 = x.view(-1, 6)  # 12/6 = 2
print(f"   x.view(-1, 6) → shape {r4.shape}  (-1 becomes 2)")

# --- flatten ---
print(f"\n{'─' * 60}")
print("📌 flatten() — collapse to 1D (common before classification layers)")
images = torch.randn(8, 3, 32, 32)  # batch of 8 color images
flat = images.view(8, -1)            # flatten each image: 3×32×32 = 3072
print(f"   Images: {images.shape} → Flattened: {flat.shape}")
print(f"   💡 This is what happens before a fully connected layer in CNNs")

# --- squeeze / unsqueeze ---
print(f"\n{'─' * 60}")
print("📌 squeeze() / unsqueeze() — remove or add size-1 dimensions")
t = torch.randn(1, 3, 1, 4)
print(f"   Original shape:         {t.shape}")
print(f"   t.squeeze() shape:      {t.squeeze().shape}      ← removed ALL size-1 dims")
print(f"   t.squeeze(0) shape:     {t.squeeze(0).shape}     ← removed dim 0 only")
print(f"   t.squeeze(2) shape:     {t.squeeze(2).shape}     ← removed dim 2 only")

s = torch.randn(3, 4)
print(f"\n   s shape:                {s.shape}")
print(f"   s.unsqueeze(0) shape:   {s.unsqueeze(0).shape}   ← added batch dimension")
print(f"   s.unsqueeze(-1) shape:  {s.unsqueeze(-1).shape}  ← added last dimension")
print(f"\n   💡 unsqueeze(0) is how you turn a single sample into a 'batch of 1'")

# --- Indexing and Slicing ---
print(f"\n{'─' * 60}")
print("📌 Indexing & Slicing (same as NumPy)")
m = torch.tensor([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 10, 11, 12]]).float()
print(f"   m =\n   {m}")
print(f"   m[0]       = {m[0]}           ← first row")
print(f"   m[:, 1]    = {m[:, 1]}           ← second column")
print(f"   m[0:2, 1:3]= {m[0:2, 1:3]}  ← rows 0-1, cols 1-2")
print(f"   m[m > 6]   = {m[m > 6]}  ← boolean indexing")

### ✍️ TODO 3: Broadcasting & Reshaping Practice

In [ ]:
# ============================================================
# TODO 3: BROADCASTING & RESHAPING EXERCISES
# ============================================================

# Exercise 3a: Create a (4, 5) tensor of random values.
#              Normalize EACH COLUMN to have mean 0 and std 1.
#              Verify with .mean(dim=0) and .std(dim=0).
#              Hint: use broadcasting — compute column means/stds first.
# Your code here:
# z = ...
# z_norm = ...
# print("Column means:", z_norm.mean(dim=0))
# print("Column stds:", z_norm.std(dim=0))


# Exercise 3b: Create a tensor of shape (24,) using arange.
#              Reshape it into ALL possible 2D shapes (e.g., 1×24, 2×12, ...).
#              Print each shape.
# Your code here:


# Exercise 3c: A model outputs predictions of shape (16, 1).
#              The targets have shape (16,).
#              This shape mismatch causes errors! Fix the predictions shape
#              using squeeze to make both (16,).
preds = torch.randn(16, 1)
targets = torch.randn(16)
# Your code here:
# preds_fixed = ...
# print(f"Predictions: {preds_fixed.shape}, Targets: {targets.shape}")


# Exercise 3d: You have 3 grayscale images of shape (28, 28) stored in a list.
#              Stack them into a single tensor of shape (3, 1, 28, 28)
#              which is (batch, channels, height, width) format.
#              Hint: torch.stack() and unsqueeze()
img1 = torch.randn(28, 28)
img2 = torch.randn(28, 28)
img3 = torch.randn(28, 28)
# Your code here:
# batch = ...
# print(f"Batch shape: {batch.shape}")

---

# Part 4: The NumPy-PyTorch Bridge 🌉

Since NumPy and PyTorch are so similar, you'll often need to convert between them. This is critical because:
- Many data loading tools return NumPy arrays
- Matplotlib plotting requires NumPy arrays
- scikit-learn works with NumPy, not PyTorch

**The important gotcha:** `torch.from_numpy()` **shares memory** with the original array. Changing one changes the other!

In [ ]:
# ============================================================
# 🌉 THE NUMPY ↔ PYTORCH BRIDGE
# ============================================================
print("=" * 60)
print("🌉 CONVERTING BETWEEN NUMPY AND PYTORCH")
print("=" * 60)

# --- NumPy → PyTorch ---
print("\n📌 NumPy → PyTorch (two ways)")
np_array = np.array([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0]])

method1 = torch.from_numpy(np_array)    # SHARES memory!
method2 = torch.tensor(np_array)        # Makes an INDEPENDENT copy

print(f"   torch.from_numpy(): {method1.shape}, dtype={method1.dtype}  ← shares memory")
print(f"   torch.tensor():     {method2.shape}, dtype={method2.dtype}  ← independent copy")

# --- PyTorch → NumPy ---
print(f"\n📌 PyTorch → NumPy")
tensor = torch.randn(2, 3)
np_back = tensor.numpy()
print(f"   tensor.numpy(): shape={np_back.shape}, type={type(np_back)}")

# --- ⚠️ SHARED MEMORY DEMO ---
print(f"\n{'─' * 60}")
print("⚠️  SHARED MEMORY GOTCHA — PAY ATTENTION!")
print("─" * 60)

original = np.array([1.0, 2.0, 3.0])
shared   = torch.from_numpy(original)
independent = torch.tensor(original)

print(f"   Before:    NumPy = {original}")
print(f"              Shared tensor  = {shared}")
print(f"              Independent    = {independent}")

original[0] = 999.0  # Change the NumPy array

print(f"\n   After changing NumPy[0] = 999:")
print(f"              NumPy = {original}")
print(f"              Shared tensor  = {shared}  ← 😱 ALSO CHANGED!")
print(f"              Independent    = {independent}  ← ✅ unaffected")

# --- GPU tensors require .cpu() first ---
print(f"\n{'─' * 60}")
print("📌 GPU tensor → NumPy (extra step needed)")
print("   GPU tensors can't convert to NumPy directly.")
print("   You must move to CPU first: tensor.cpu().numpy()")
if torch.cuda.is_available():
    gpu_tensor = torch.randn(3, device='cuda')
    np_from_gpu = gpu_tensor.cpu().numpy()
    print(f"   gpu_tensor.cpu().numpy() = {np_from_gpu}")
else:
    print("   (Skipped — no GPU available, but remember this for later!)")

print(f"\n💡 RULE OF THUMB:")
print(f"   • Need a copy? Use torch.tensor(np_array)")
print(f"   • Want efficiency & shared memory? Use torch.from_numpy(np_array)")

---

# Part 5: GPU Acceleration 🚀

GPUs (Graphics Processing Units) can perform thousands of matrix operations in parallel, making them **10-100x faster** than CPUs for deep learning workloads.

**How it works in PyTorch:**
- Check if a GPU is available: `torch.cuda.is_available()`
- Move tensors to GPU: `tensor.to('cuda')` or `tensor.cuda()`
- Move back to CPU: `tensor.to('cpu')` or `tensor.cpu()`
- Create directly on GPU: `torch.randn(3, 3, device='cuda')`

**The golden rule:** All tensors in an operation must be on the **same device**. Mixing CPU and GPU tensors causes an error.

In [ ]:
# ============================================================
# 🚀 GPU ACCELERATION
# ============================================================
print("=" * 60)
print("🚀 GPU ACCELERATION")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n   Using device: {device}")

# --- Moving tensors between devices ---
print(f"\n📌 Moving tensors to a device")
x = torch.randn(3, 3)
print(f"   Created on: {x.device}")

x_on_device = x.to(device)
print(f"   Moved to:   {x_on_device.device}")

# Create directly on device
y = torch.randn(3, 3, device=device)
print(f"   Created directly on: {y.device}")

# Operations — both must be on the same device!
z = x_on_device + y
print(f"   x + y computed on: {z.device}")

# --- Device mismatch error (common bug!) ---
print(f"\n{'─' * 60}")
print("⚠️  COMMON BUG: Device mismatch")
if torch.cuda.is_available():
    cpu_tensor = torch.randn(3)
    gpu_tensor = torch.randn(3, device='cuda')
    try:
        result = cpu_tensor + gpu_tensor  # This will fail!
    except RuntimeError as e:
        print(f"   Error: {e}")
    print(f"   Fix: Move both to the same device first!")
    print(f"   result = cpu_tensor.to('cuda') + gpu_tensor  ✅")
else:
    print("   (On CPU — but when using GPU, always ensure tensors are on the same device)")
    print("   Mixing CPU tensor + GPU tensor → RuntimeError!")

# --- Speed comparison ---
print(f"\n{'─' * 60}")
print("📌 Speed comparison: CPU vs GPU")
import time

sizes = [1000, 3000]
for size in sizes:
    a_cpu = torch.randn(size, size)
    b_cpu = torch.randn(size, size)

    start = time.time()
    _ = a_cpu @ b_cpu
    cpu_time = time.time() - start

    if torch.cuda.is_available():
        a_gpu = a_cpu.to('cuda')
        b_gpu = b_cpu.to('cuda')
        torch.cuda.synchronize()
        start = time.time()
        _ = a_gpu @ b_gpu
        torch.cuda.synchronize()
        gpu_time = time.time() - start
        speedup = cpu_time / gpu_time
        print(f"   {size}×{size} matmul:  CPU={cpu_time:.4f}s  GPU={gpu_time:.4f}s  ({speedup:.1f}x faster)")
    else:
        print(f"   {size}×{size} matmul:  CPU={cpu_time:.4f}s  (no GPU to compare)")

print(f"\n   💡 The bigger the computation, the bigger the GPU advantage!")
print(f"   Small tensors may be SLOWER on GPU due to transfer overhead.")

### ✍️ TODO 4: NumPy Bridge & Device Practice

In [ ]:
# ============================================================
# TODO 4: NUMPY BRIDGE & DEVICE EXERCISES
# ============================================================

# Exercise 4a: Create a NumPy array of shape (3, 4) with values 1-12.
#              Convert it to a PyTorch tensor using BOTH methods.
#              Modify the NumPy array and show which tensor changed.
# Your code here:


# Exercise 4b: Create a tensor on the best available device (GPU if available,
#              else CPU). Print which device it's on.
#              Then convert it to a NumPy array (remember: GPU needs .cpu() first!)
# Hint: use the 'device' variable we defined earlier
# Your code here:


# Exercise 4c: Write a function that takes a NumPy array, converts it to a
#              PyTorch tensor on the best device, doubles all values,
#              and returns the result as a NumPy array.
# def numpy_double_via_pytorch(np_arr):
#     ...
#     return result_np
# Test it:
# test = np.array([1.0, 2.0, 3.0, 4.0])
# print(numpy_double_via_pytorch(test))  # Should be [2.0, 4.0, 6.0, 8.0]

---

# Part 6: Autograd — The Engine Behind Learning 🪄

This is where PyTorch becomes fundamentally different from NumPy. **Autograd** (automatic differentiation) is the secret sauce that makes deep learning possible.

## The Core Idea

Remember gradient descent from the Regression notebook? We computed derivatives by hand to figure out which direction to adjust our parameters. With autograd, **PyTorch computes all the derivatives for you automatically**, no matter how complex your model is.

**The analogy:** Imagine you're baking a cake with 50 ingredients. If the cake tastes bad, you need to know *how much each ingredient contributed* to the problem. Autograd is like having a magical kitchen assistant who tracks the effect of every ingredient and tells you exactly how to adjust each one.

$$\text{gradient} = \frac{\partial \text{loss}}{\partial \text{parameter}}$$

**How it works in 3 steps:**
1. **Mark** tensors with `requires_grad=True` (these are your learnable parameters)
2. **Compute** — run any operations (the forward pass)
3. **Call `.backward()`** on the result → gradients appear in `.grad`

In [ ]:
# ============================================================
# 🪄 AUTOGRAD — AUTOMATIC GRADIENT COMPUTATION
# ============================================================
print("=" * 60)
print("🪄 AUTOGRAD — THE ENGINE BEHIND LEARNING")
print("=" * 60)

# --- Basic gradient computation ---
print("\n📌 Step 1: Mark a tensor to track gradients")
w = torch.tensor([3.0], requires_grad=True)
print(f"   w = {w.item()}, requires_grad = {w.requires_grad}")

print("\n📌 Step 2: Perform computation (forward pass)")
# Let's compute f(w) = (w - 7)²
# We know the derivative: df/dw = 2(w - 7) = 2(3 - 7) = -8
loss = (w - 7) ** 2
print(f"   f(w) = (w - 7)² = ({w.item()} - 7)² = {loss.item()}")

print("\n📌 Step 3: Call .backward() to compute gradients")
loss.backward()
print(f"   Gradient df/dw = {w.grad.item()}")
print(f"   Manual check:  2 × (3 - 7) = {2 * (3 - 7)}")
print(f"   ✅ They match!")

print(f"\n{'─' * 60}")
print("📌 What the gradient tells us:")
print(f"   w.grad = {w.grad.item()} (negative)")
print(f"   → Negative gradient means: increasing w will DECREASE loss")
print(f"   → So gradient descent moves w in the positive direction (toward 7)")
print(f"   → w_new = w - lr × grad = 3 - 0.1 × ({w.grad.item()}) = {3 - 0.1 * w.grad.item()}")
print(f"   → We moved closer to the minimum at w = 7! ✅")

# --- Multiple variables ---
print(f"\n{'─' * 60}")
print("📌 Autograd works with multiple variables too!")
x = torch.tensor([1.0], requires_grad=True)
y = torch.tensor([2.0], requires_grad=True)

# f(x, y) = x² + 3xy + y²
f = x**2 + 3*x*y + y**2
f.backward()
print(f"   f(x, y) = x² + 3xy + y²")
print(f"   At x={x.item()}, y={y.item()}:")
print(f"   df/dx = 2x + 3y = {x.grad.item()}  (manual: {2*1 + 3*2})")
print(f"   df/dy = 3x + 2y = {y.grad.item()}  (manual: {3*1 + 2*2})")

# --- Important: zero gradients! ---
print(f"\n{'─' * 60}")
print("⚠️  CRITICAL: Always zero gradients before computing new ones!")
print("   PyTorch ACCUMULATES gradients by default (adds to existing)")
print(f"   Before zero: w.grad = {w.grad.item()}")
w.grad.zero_()
print(f"   After zero:  w.grad = {w.grad.item()}")
print(f"\n   💡 Forgetting to zero gradients is one of the most common PyTorch bugs!")

In [ ]:
# ============================================================
# 🔒 CONTROLLING GRADIENT TRACKING
# ============================================================
print("=" * 60)
print("🔒 CONTROLLING GRADIENT TRACKING")
print("=" * 60)

# --- torch.no_grad() context ---
print("\n📌 torch.no_grad() — disable gradient tracking")
print("   Use this during evaluation or when updating parameters manually")

w = torch.tensor([5.0], requires_grad=True)

# With gradient tracking (default)
y1 = w * 3
print(f"   With grad:    y1.requires_grad = {y1.requires_grad}")

# Without gradient tracking
with torch.no_grad():
    y2 = w * 3
    print(f"   Without grad: y2.requires_grad = {y2.requires_grad}")

print(f"\n   💡 torch.no_grad() is used in two places:")
print(f"   1. During evaluation (no learning, so no gradients needed)")
print(f"   2. When manually updating parameters (don't track the update itself)")

# --- detach() — disconnect from computation graph ---
print(f"\n{'─' * 60}")
print("📌 .detach() — create a tensor disconnected from the graph")
a = torch.tensor([2.0], requires_grad=True)
b = a * 3       # b is connected to a in the computation graph
c = b.detach()   # c has the same value but is disconnected

print(f"   a.requires_grad = {a.requires_grad}")
print(f"   b.requires_grad = {b.requires_grad}  (connected to a)")
print(f"   c.requires_grad = {c.requires_grad} (detached copy)")

print(f"\n   💡 .detach() is useful when you want to use a value")
print(f"   without affecting gradient computation (e.g., logging metrics)")

In [ ]:
# ============================================================
# 📉 GRADIENT DESCENT FROM SCRATCH WITH AUTOGRAD
# ============================================================
print("=" * 60)
print("📉 WATCHING GRADIENT DESCENT IN ACTION")
print("=" * 60)
print("\n   Goal: Find w that minimizes f(w) = (w - 7)²")
print("   Starting at w = 0.0, learning rate = 0.1\n")

w = torch.tensor([0.0], requires_grad=True)
lr = 0.1
history = {'w': [], 'loss': []}

print(f"   {'Step':>4}  {'w':>8}  {'loss':>10}  {'gradient':>10}  {'update':>10}")
print(f"   {'─'*50}")

for step in range(25):
    # Forward pass — compute the loss
    loss = (w - 7) ** 2

    # Record for plotting
    history['w'].append(w.item())
    history['loss'].append(loss.item())

    # Backward pass — compute gradients
    loss.backward()

    if step % 5 == 0 or step < 3:
        print(f"   {step:4d}  {w.item():8.4f}  {loss.item():10.4f}  {w.grad.item():10.4f}  {(-lr * w.grad.item()):10.4f}")

    # Update parameters (no gradient tracking!)
    with torch.no_grad():
        w -= lr * w.grad

    # Zero gradients for next iteration
    w.grad.zero_()

print(f"\n   ✅ Converged! w = {w.item():.4f} (target was 7.0)")

# --- Visualize the optimization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss over steps
axes[0].plot(history['loss'], 'b-o', markersize=3, linewidth=2)
axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss Decreasing Over Training Steps', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Plot 2: Optimization path on loss landscape
w_range = np.linspace(-1, 9, 100)
loss_landscape = (w_range - 7) ** 2
axes[1].plot(w_range, loss_landscape, 'b-', linewidth=2, label='f(w) = (w-7)²')
axes[1].scatter(history['w'], history['loss'], c=range(len(history['w'])),
               cmap='Reds', s=50, zorder=5, edgecolors='black', linewidth=0.5)
axes[1].scatter([7], [0], color='green', s=200, marker='*', zorder=6, label='Minimum (w=7)')
axes[1].set_xlabel('w', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Gradient Descent on Loss Landscape', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n   💡 Red dots show the optimization path — notice how steps get smaller near the minimum!")

In [ ]:
# ============================================================
# 📉 MULTI-VARIABLE GRADIENT DESCENT
# ============================================================
print("=" * 60)
print("📉 GRADIENT DESCENT WITH MULTIPLE PARAMETERS")
print("=" * 60)
print("\n   Goal: Find (a, b) that minimizes f(a,b) = (a - 3)² + 10·(b - 2)²")
print("   This is an elongated bowl — harder than a simple parabola!\n")

a = torch.tensor([-5.0], requires_grad=True)
b = torch.tensor([-3.0], requires_grad=True)
lr = 0.05

path_a, path_b, losses = [], [], []

for step in range(100):
    loss = (a - 3)**2 + 10*(b - 2)**2

    path_a.append(a.item())
    path_b.append(b.item())
    losses.append(loss.item())

    loss.backward()

    with torch.no_grad():
        a -= lr * a.grad
        b -= lr * b.grad

    a.grad.zero_()
    b.grad.zero_()

    if step % 20 == 0 or step == 99:
        print(f"   Step {step:>3d}  │  a={a.item():7.3f}  b={b.item():7.3f}  │  loss={loss.item():.4f}")

print(f"\n   ✅ Converged to a={a.item():.3f}, b={b.item():.3f} (target: 3, 2)")

# --- Visualize ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot with optimization path
aa, bb = np.meshgrid(np.linspace(-6, 7, 100), np.linspace(-5, 6, 100))
zz = (aa - 3)**2 + 10*(bb - 2)**2
axes[0].contour(aa, bb, zz, levels=30, cmap='coolwarm', alpha=0.6)
axes[0].contourf(aa, bb, zz, levels=30, cmap='coolwarm', alpha=0.2)
axes[0].plot(path_a, path_b, 'r-o', markersize=2, linewidth=1.5, label='Optimization path')
axes[0].scatter([path_a[0]], [path_b[0]], color='red', s=100, marker='s',
               edgecolors='black', zorder=5, label='Start')
axes[0].scatter([3], [2], color='gold', s=200, marker='*', edgecolors='black',
               linewidth=1.5, zorder=6, label='Minimum (3, 2)')
axes[0].set_xlabel('a', fontsize=12)
axes[0].set_ylabel('b', fontsize=12)
axes[0].set_title('Optimization Path on Loss Landscape', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss curve
axes[1].plot(losses, 'b-', linewidth=2)
axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Loss Over Steps', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n   💡 Notice the path curves — gradient descent takes different-sized steps")
print("   along each dimension based on how steep the landscape is!")

In [ ]:
# ============================================================
# 📈 PUTTING IT ALL TOGETHER: LINEAR REGRESSION FROM SCRATCH
# ============================================================
print("=" * 60)
print("📈 LINEAR REGRESSION USING ONLY TENSORS + AUTOGRAD")
print("=" * 60)
print("\n   No nn.Module, no optimizer — just tensors and gradients!")
print("   This shows you exactly what happens 'under the hood'.\n")

# Generate synthetic data: y = 3x + 7 + noise
torch.manual_seed(42)
X = torch.linspace(-3, 3, 100).unsqueeze(1)   # (100, 1)
y_true = 3 * X + 7 + torch.randn(100, 1) * 0.8

# Initialize parameters (the values we want to LEARN)
w = torch.tensor([[0.0]], requires_grad=True)   # weight (slope)
b = torch.tensor([0.0], requires_grad=True)    # bias (intercept)
lr = 0.05

print(f"   Data: {X.shape[0]} points, true relationship: y = 3x + 7")
print(f"   Starting parameters: w = {w.item():.1f}, b = {b.item():.1f}")
print(f"   Learning rate: {lr}\n")

history = {'loss': [], 'w': [], 'b': []}

print(f"   {'Epoch':>5}  {'Loss':>8}  {'w':>8}  {'b':>8}  {'w target':>10}  {'b target':>10}")
print(f"   {'─'*55}")

for epoch in range(200):
    # Forward: predict
    y_pred = X @ w + b                         # (100,1) @ (1,1) + (1,) = (100,1)

    # Compute loss: mean squared error
    loss = ((y_pred - y_true) ** 2).mean()

    # Backward: compute gradients
    loss.backward()

    # Update: gradient descent (no tracking!)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()

    history['loss'].append(loss.item())
    history['w'].append(w.item())
    history['b'].append(b.item())

    if epoch % 40 == 0 or epoch == 199:
        print(f"   {epoch:5d}  {loss.item():8.4f}  {w.item():8.4f}  {b.item():8.4f}  {'3.0':>10}  {'7.0':>10}")

print(f"\n   ✅ Learned: y = {w.item():.3f}x + {b.item():.3f}")
print(f"   Target:   y = 3.000x + 7.000")

# --- Visualize ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Data + learned line
axes[0].scatter(X.numpy(), y_true.numpy(), s=10, alpha=0.5, label='Data')
x_line = torch.linspace(-3, 3, 100).unsqueeze(1)
with torch.no_grad():
    y_line = x_line @ w + b
axes[0].plot(x_line.numpy(), y_line.numpy(), 'r-', linewidth=2, label=f'Learned: y={w.item():.2f}x+{b.item():.2f}')
axes[0].plot(x_line.numpy(), (3*x_line + 7).numpy(), 'g--', linewidth=2, alpha=0.5, label='True: y=3x+7')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('y', fontsize=12)
axes[0].set_title('Data and Learned Line', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss curve
axes[1].plot(history['loss'], linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MSE Loss', fontsize=12)
axes[1].set_title('Loss Over Training', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Parameter convergence
axes[2].plot(history['w'], label='w (learned)', linewidth=2)
axes[2].axhline(y=3, color='r', linestyle='--', alpha=0.5, label='w (true) = 3')
axes[2].plot(history['b'], label='b (learned)', linewidth=2)
axes[2].axhline(y=7, color='g', linestyle='--', alpha=0.5, label='b (true) = 7')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Parameter Value', fontsize=12)
axes[2].set_title('Parameters Converging to True Values', fontsize=14)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n   💡 We just trained a model with ONLY tensors and autograd!")
print("   No nn.Linear, no optimizer — this is what they do internally.")
print("   In the next session, you'll see how nn.Module wraps this up neatly.")

### ✍️ TODO 5: Autograd Practice

In [ ]:
# ============================================================
# TODO 5: AUTOGRAD EXERCISES
# ============================================================

# Exercise 5a: Compute the gradient of f(x) = x³ + 2x² - 5x + 3 at x = 2.
#              The analytical derivative is f'(x) = 3x² + 4x - 5.
#              Verify that autograd gives the same answer.
# Your code here:
# x = torch.tensor([2.0], requires_grad=True)
# f = ...
# f.backward()
# print(f"Autograd gradient: {x.grad.item()}")
# print(f"Manual gradient:   {3*4 + 4*2 - 5}")


# Exercise 5b: Use autograd to find the minimum of f(x) = (x - 5)² + (x - 5)⁴
#              Start at x = 0, learning rate = 0.01, run for 200 steps.
#              Print the final x value (should be close to 5.0).
# Your code here:


# Exercise 5c: Implement gradient descent to fit y = ax² + bx + c to this data:
#              x_data = [-2, -1, 0, 1, 2], y_data = [7, 2, 1, 4, 11]
#              The true function is y = 2x² - x + 1.
#              Use 500 steps with lr = 0.01.
# Hint: Create a, b, c as tensors with requires_grad=True
x_data = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
y_data = torch.tensor([7.0, 2.0, 1.0, 4.0, 11.0])
# Your code here:
# a = torch.tensor([0.0], requires_grad=True)
# b = torch.tensor([0.0], requires_grad=True)
# c = torch.tensor([0.0], requires_grad=True)
# ...
# print(f"Learned: y = {a.item():.3f}x² + {b.item():.3f}x + {c.item():.3f}")
# print(f"True:    y = 2.000x² + -1.000x + 1.000")


# Exercise 5d (CHALLENGE): Learning rate experiment.
#              Using f(w) = (w - 7)² starting from w = 0:
#              Run gradient descent with lr = 0.01, 0.1, 0.5, and 1.0 for 30 steps.
#              Plot all four loss curves on the same chart.
#              What happens when lr is too large? Too small?
# Your code here:

---

# Part 7: Common Patterns & Gotchas 🚧

Before you move on, here are the most common mistakes beginners make and the patterns you'll use every day.

In [ ]:
# ============================================================
# 🚧 COMMON PATTERNS & GOTCHAS
# ============================================================
print("=" * 60)
print("🚧 COMMON PATTERNS & GOTCHAS")
print("=" * 60)

# --- In-place operations (be careful!) ---
print("\n📌 Gotcha 1: In-place operations modify the original tensor")
original = torch.tensor([1.0, 2.0, 3.0])
view = original[:]  # This is a VIEW, not a copy!
view[0] = 999.0
print(f"   After changing view[0] = 999:")
print(f"   original = {original}  ← also changed!")
print(f"   Fix: use .clone() to make an independent copy")
safe_copy = original.clone()
safe_copy[0] = -1.0
print(f"   After changing clone[0] = -1:")
print(f"   original = {original}  ← unchanged ✅")

# --- Tensor vs tensor (float vs int) ---
print(f"\n{'─' * 60}")
print("📌 Gotcha 2: Integer vs Float tensors")
int_t = torch.tensor([1, 2, 3])
try:
    result = int_t / 2
    print(f"   [1, 2, 3] / 2 = {result}  (PyTorch auto-converts to float)")
except:
    print("   Error with integer division!")
print(f"   But: torch.tensor([1, 2, 3]) + 0.5 = {torch.tensor([1, 2, 3]) + 0.5}")
print(f"   💡 ML almost always uses float32 — add .float() when in doubt")

# --- Contiguous memory ---
print(f"\n{'─' * 60}")
print("📌 Gotcha 3: Contiguous tensors and .view() vs .reshape()")
t = torch.tensor([[1, 2, 3], [4, 5, 6]])
t_transposed = t.T  # Transpose makes it non-contiguous
print(f"   t.T is contiguous? {t_transposed.is_contiguous()}")
try:
    t_transposed.view(-1)  # This may fail!
except RuntimeError as e:
    print(f"   t.T.view(-1) fails: {e}")
print(f"   Fix: t.T.contiguous().view(-1) = {t_transposed.contiguous().view(-1)}")
print(f"   Or:  t.T.reshape(-1)           = {t_transposed.reshape(-1)}  ← reshape handles it")
print(f"   💡 reshape() is safer than view() — use it when unsure")

# --- Gradient accumulation ---
print(f"\n{'─' * 60}")
print("📌 Gotcha 4: Gradient accumulation")
w = torch.tensor([2.0], requires_grad=True)

loss1 = (w * 3)
loss1.backward()
print(f"   After 1st backward: w.grad = {w.grad.item()}")

loss2 = (w * 3)
loss2.backward()
print(f"   After 2nd backward: w.grad = {w.grad.item()}  ← accumulated! (3 + 3 = 6)")

w.grad.zero_()
loss3 = (w * 3)
loss3.backward()
print(f"   After zero + backward: w.grad = {w.grad.item()}  ← correct ✅")
print(f"   💡 ALWAYS call .grad.zero_() or optimizer.zero_grad() before .backward()")

### ✍️ TODO 6: Comprehensive Review Challenge

In [ ]:
# ============================================================
# TODO 6: COMPREHENSIVE REVIEW — PUT IT ALL TOGETHER
# ============================================================

# Exercise 6a: The Data Pipeline Challenge
#              Given this raw NumPy data, prepare it for PyTorch:
#              1. Convert to PyTorch float32 tensors
#              2. Normalize features to mean=0, std=1 (use broadcasting!)
#              3. Move to the best available device
#              4. Print the final shape, dtype, and device
raw_features = np.array([[170, 65, 25],
                          [180, 80, 30],
                          [155, 50, 22],
                          [190, 90, 35],
                          [165, 60, 27]], dtype=np.float32)
raw_labels = np.array([0, 1, 0, 1, 0], dtype=np.float32)
# Your code here:


# Exercise 6b: Gradient Descent Race
#              Implement gradient descent on f(x) = x⁴ - 3x³ + 2 starting
#              from x = -1.0. Try two learning rates: 0.01 and 0.001.
#              Run 500 steps each and plot BOTH loss curves on one chart.
#              Which learning rate converges faster?
# Your code here:


# Exercise 6c (CHALLENGE): Multi-feature Linear Regression
#              Generate data where y = 2*x₁ + 3*x₂ - x₃ + 5 + noise
#              Use 200 samples, 3 features.
#              Learn w (shape 3×1) and b (scalar) using gradient descent.
#              Print the learned vs true parameters.
# Hints:
#   X = torch.randn(200, 3)
#   y = X @ true_w + true_b + noise
#   Prediction: y_pred = X @ w + b
# Your code here:

---

# Part 8: Summary & Quick Reference Card 📝

You've now mastered the foundations of PyTorch! Everything in deep learning — from simple classifiers to billion-parameter language models — is built on these exact concepts.

## What You've Learned

| Concept | What It Does | Key Functions |
|:---|:---|:---|
| **Tensors** | Store data (like NumPy arrays with superpowers) | `torch.tensor()`, `.zeros()`, `.randn()` |
| **Attributes** | Debug shape/type/device issues | `.shape`, `.dtype`, `.device` |
| **Operations** | Compute with tensors | `+`, `*`, `@`, `.sum()`, `.mean()` |
| **Broadcasting** | Auto-match shapes in operations | Same rules as NumPy |
| **Reshaping** | Reorganize tensor dimensions | `.view()`, `.reshape()`, `.squeeze()` |
| **NumPy Bridge** | Convert between frameworks | `torch.from_numpy()`, `.numpy()` |
| **GPU** | Accelerate computation | `.to(device)`, `.cuda()`, `.cpu()` |
| **Autograd** | Compute gradients automatically | `requires_grad=True`, `.backward()`, `.grad` |
| **Gradient Descent** | Learn parameters from data | `w -= lr * w.grad` |

**Coming up next:** Karthik will show you how to build neural networks using `nn.Module`, `nn.Linear`, loss functions, optimizers, and the training loop — all powered by the autograd engine you just learned!

## 📝 PyTorch Quick Reference Card

### Tensor Creation
| Code | Description |
|:---|:---|
| `torch.tensor([1, 2, 3])` | From Python list |
| `torch.zeros(3, 4)` | All zeros |
| `torch.ones(2, 3)` | All ones |
| `torch.randn(3, 3)` | Normal distribution |
| `torch.rand(3, 3)` | Uniform [0, 1) |
| `torch.arange(0, 10, 2)` | Range |
| `torch.eye(3)` | Identity matrix |
| `torch.full((2, 3), 7.0)` | Filled with value |

### Tensor Info (Check These Constantly!)
| Code | Description |
|:---|:---|
| `tensor.shape` | Dimensions |
| `tensor.dtype` | Data type (float32) |
| `tensor.device` | CPU or GPU |
| `tensor.requires_grad` | Tracks gradients? |

### NumPy Bridge
| Code | Description |
|:---|:---|
| `torch.from_numpy(arr)` | Shared memory! |
| `torch.tensor(arr)` | Independent copy |
| `tensor.numpy()` | To NumPy (CPU only) |
| `gpu_tensor.cpu().numpy()` | GPU to NumPy |

### Operations
| Code | Description |
|:---|:---|
| `a + b`, `a * b`, `a ** 2` | Element-wise |
| `A @ B` / `torch.matmul(A, B)` | Matrix multiplication |
| `tensor.sum(dim=0)` | Sum along axis |
| `tensor.mean()`, `.std()`, `.max()` | Reductions |
| `tensor > 5` | Boolean mask |

### Reshaping
| Code | Description |
|:---|:---|
| `tensor.view(3, 4)` | Reshape (contiguous) |
| `tensor.reshape(3, -1)` | Reshape (flexible) |
| `tensor.squeeze()` | Remove size-1 dims |
| `tensor.unsqueeze(0)` | Add dim at position 0 |
| `tensor.flatten()` | Flatten to 1D |
| `tensor.T` | Transpose |

### GPU
| Code | Description |
|:---|:---|
| `tensor.to("cuda")` | Move to GPU |
| `tensor.to("cpu")` | Move to CPU |
| `tensor.cuda()` / `.cpu()` | Shorthand |
| `torch.randn(3, device="cuda")` | Create on GPU |

### Autograd
| Code | Description |
|:---|:---|
| `requires_grad=True` | Track this tensor |
| `loss.backward()` | Compute all gradients |
| `tensor.grad` | Read the gradient |
| `tensor.grad.zero_()` | Reset gradient to 0 |
| `with torch.no_grad():` | Disable tracking |
| `tensor.detach()` | Disconnect from graph |

### Gradient Descent Pattern
```python
loss = compute_loss(params)      # Forward pass
loss.backward()                  # Compute gradients
with torch.no_grad():
    params -= lr * params.grad   # Update parameters
params.grad.zero_()              # Reset for next step
```

---

### 🚀 What's Next

You now have the PyTorch foundation! In the next session, Karthik will build on this to show you:

1. **nn.Module** — wrapping layers into reusable models
2. **Loss functions** — MSELoss, CrossEntropyLoss, etc.
3. **Optimizers** — Adam, SGD (automate the update step!)
4. **The training loop** — the 5-step pattern for all DL
5. **DataLoader** — efficient batched data loading
6. **Model saving & loading**

Everything above uses the tensors, operations, and autograd you just learned — you're ready!

---

## 🎯 Mastery Checklist

Before moving on, make sure you can:

- [ ] Create tensors from lists, NumPy arrays, and special constructors (`zeros`, `randn`, `eye`, etc.)
- [ ] Check the 3 key attributes: `.shape`, `.dtype`, `.device`
- [ ] Perform element-wise operations, reductions (`sum`, `mean`), and understand `dim`
- [ ] Do matrix multiplication with `@` and explain why it matters for ML
- [ ] Use broadcasting to normalize a dataset without manual loops
- [ ] Reshape tensors with `.view()`, `.reshape()`, `.squeeze()`, and `.unsqueeze()`
- [ ] Convert between NumPy and PyTorch (and know when memory is shared!)
- [ ] Move tensors to GPU and back, and handle device mismatch errors
- [ ] Explain what `requires_grad=True` does and how `.backward()` computes gradients
- [ ] Implement gradient descent from scratch using autograd
- [ ] Remember to zero gradients before each `.backward()` call

**If you checked all boxes — you're ready for neural networks!**

---

*This notebook is part of the Machine Learning 101 series. It builds on the NumPy notebook and prepares you for the neural networks session.*